# 02 — Exploratory Data Analysis
Understand document types, handle duplicates, decide which financial columns are usable,
and quarterize cumulative income-statement figures. Outputs `statements_quarterized.csv`.

In [21]:
import pandas as pd
import numpy as np


## 1. Load processed data

In [22]:
stock_prices = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedstock_prices_clean.csv", parse_dates=["Date"])
financials   = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedfinancials_clean.csv",   parse_dates=["Date", "DisclosedDate"])

print("stock_prices:", stock_prices.shape)
print("financials  :", financials.shape)

stock_prices: (2332531, 12)
financials  : (92956, 45)


C:\Users\naksh\AppData\Local\Temp\ipykernel_1728\2061366380.py:2: DtypeWarning: Columns (0: OrdinaryProfit, 1: Profit, 2: EarningsPerShare, 3: TotalAssets, 4: Equity, 5: EquityToAssetRatio, 6: NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock, 7: AverageNumberOfShares) have mixed types. Specify dtype option on import or set low_memory=False.
  financials   = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedfinancials_clean.csv",   parse_dates=["Date", "DisclosedDate"])


## 2. Document-type analysis

In [23]:
print(financials["TypeOfDocument"].value_counts())
print("\nAccounting standard breakdown:")
print(financials["TypeOfDocument"].str.extract(r"(JP|IFRS|US)")[0].value_counts())

TypeOfDocument
ForecastRevision                                     16995
FYFinancialStatements_Consolidated_JP                14780
3QFinancialStatements_Consolidated_JP                14728
1QFinancialStatements_Consolidated_JP                14718
2QFinancialStatements_Consolidated_JP                14624
NumericalCorrection                                   2551
FYFinancialStatements_NonConsolidated_JP              2458
3QFinancialStatements_NonConsolidated_JP              2411
1QFinancialStatements_NonConsolidated_JP              2405
2QFinancialStatements_NonConsolidated_JP              2366
2QFinancialStatements_Consolidated_IFRS                962
1QFinancialStatements_Consolidated_IFRS                960
FYFinancialStatements_Consolidated_IFRS                897
3QFinancialStatements_Consolidated_IFRS                881
FYFinancialStatements_REIT                             571
ForecastRevision_REIT                                  320
FYFinancialStatements_Consolidated_US    

In [24]:
# Keep only JP-Consolidated quarterly/annual statements and forecast revisions
KEEP_DOCS = [
    "ForecastRevision",
    "1QFinancialStatements_Consolidated_JP",
    "2QFinancialStatements_Consolidated_JP",
    "3QFinancialStatements_Consolidated_JP",
    "FYFinancialStatements_Consolidated_JP",
]

price_codes = set(stock_prices["SecuritiesCode"].unique())

financials = (
    financials[
        financials["TypeOfDocument"].isin(KEEP_DOCS) &
        financials["SecuritiesCode"].isin(price_codes)
    ]
    .sort_values(["SecuritiesCode", "Date"])
    .copy()
)

print("After filtering:", financials.shape)

After filtering: (40968, 45)


## 3. Numeric columns — cast & inspect

In [25]:
NUMERIC_COLS = [
    # Income statement
    "NetSales", "OperatingProfit", "OrdinaryProfit", "Profit",
    # Balance sheet
    "TotalAssets", "Equity",
    # Per-share
    "BookValuePerShare", "EarningsPerShare",
    # Forecasts
    "ForecastNetSales", "ForecastOperatingProfit", "ForecastOrdinaryProfit",
    "ForecastProfit", "ForecastEarningsPerShare",
    # Share counts
    "AverageNumberOfShares",
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock",
    "NumberOfTreasuryStockAtTheEndOfFiscalYear",
    # Ratios provided
    "EquityToAssetRatio",
]

DIVIDEND_COLS = [
    "ResultDividendPerShare1stQuarter",  "ResultDividendPerShare2ndQuarter",
    "ResultDividendPerShare3rdQuarter",  "ResultDividendPerShareFiscalYearEnd",
    "ResultDividendPerShareAnnual",
    "ForecastDividendPerShare1stQuarter", "ForecastDividendPerShare2ndQuarter",
    "ForecastDividendPerShare3rdQuarter", "ForecastDividendPerShareFiscalYearEnd",
    "ForecastDividendPerShareAnnual",
]

financials[NUMERIC_COLS]  = financials[NUMERIC_COLS].apply(pd.to_numeric,  errors="coerce")
financials[DIVIDEND_COLS] = financials[DIVIDEND_COLS].apply(pd.to_numeric, errors="coerce")

print("Missing rates for core numeric columns:")
print(financials[NUMERIC_COLS].isna().mean().sort_values(ascending=False))

Missing rates for core numeric columns:
BookValuePerShare                                                               0.616750
OperatingProfit                                                                 0.231913
NumberOfTreasuryStockAtTheEndOfFiscalYear                                       0.215363
OrdinaryProfit                                                                  0.201377
NetSales                                                                        0.201084
NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock    0.200986
AverageNumberOfShares                                                           0.200986
EquityToAssetRatio                                                              0.200986
EarningsPerShare                                                                0.200986
Profit                                                                          0.200962
TotalAssets                                                           

## 4. Split into statements vs. forecast revisions

In [26]:
STATEMENT_DOCS = [
    "1QFinancialStatements_Consolidated_JP",
    "2QFinancialStatements_Consolidated_JP",
    "3QFinancialStatements_Consolidated_JP",
    "FYFinancialStatements_Consolidated_JP",
]

statements = financials[financials["TypeOfDocument"].isin(STATEMENT_DOCS)].copy()
forecasts  = financials[financials["TypeOfDocument"] == "ForecastRevision"].copy()

print("statements:", statements.shape)
print("forecasts :", forecasts.shape)

statements: (32738, 45)
forecasts : (8230, 45)


## 5. Quarterize cumulative income-statement values
JPX reports are *cumulative* (Q2 = H1 total). We convert to true quarterly increments.

In [27]:
QUARTER_MAP = {
    "1QFinancialStatements_Consolidated_JP": 1,
    "2QFinancialStatements_Consolidated_JP": 2,
    "3QFinancialStatements_Consolidated_JP": 3,
    "FYFinancialStatements_Consolidated_JP": 4,
}

statements["quarter"]     = statements["TypeOfDocument"].map(QUARTER_MAP)
statements["fiscal_year"] = pd.to_datetime(statements["CurrentPeriodEndDate"]).dt.year

statements = statements.sort_values(["SecuritiesCode", "fiscal_year", "quarter"])

FLOW_COLS = ["NetSales", "OperatingProfit", "OrdinaryProfit", "Profit"]

prev_quarter = (
    statements
    .groupby(["SecuritiesCode", "fiscal_year"])["quarter"]
    .shift(1)
)

for col in FLOW_COLS:
    statements[f"q_{col}"] = np.nan

    q1_mask    = statements["quarter"] == 1
    valid_mask = (statements["quarter"] > 1) & (prev_quarter == statements["quarter"] - 1)

    statements.loc[q1_mask,    f"q_{col}"] = statements.loc[q1_mask, col]
    statements.loc[valid_mask, f"q_{col}"] = (
        statements.loc[valid_mask, col]
        - statements.groupby(["SecuritiesCode", "fiscal_year"])[col].shift(1).loc[valid_mask]
    )

print("Quarterized columns missing rates:")
print(statements[[f"q_{c}" for c in FLOW_COLS]].isna().mean())

Quarterized columns missing rates:
q_NetSales           0.095272
q_OperatingProfit    0.134431
q_OrdinaryProfit     0.095791
q_Profit             0.095088
dtype: float64


In [28]:
###############################################################################
# Trailing Twelve Month (TTM) Features
###############################################################################

ttm_base = [
    "q_NetSales",
    "q_OperatingProfit",
    "q_OrdinaryProfit",
    "q_Profit",
]

statements = statements.sort_values(
    ["SecuritiesCode", "DisclosedDate"]
)

for col in ttm_base:

    statements[f"{col}_ttm"] = (
        statements
        .groupby("SecuritiesCode")[col]
        .transform(
            lambda x: x.rolling(
                4,
                min_periods=4
            ).sum()
        )
    )

In [29]:
statements[["NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock", "EarningsPerShare"]]=financials[["NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock", "EarningsPerShare"]]

## 6. Save

In [30]:
statements.to_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\statements_quarterized.csv", index=False)
print("Saved: statements_quarterized.csv —", statements.shape)

Saved: statements_quarterized.csv — (32738, 55)
